# Exportar município → GeoPackage único

Consolida todos os dados processados de um município em um único `.gpkg` pronto para o QGIS.

**Fontes lidas (sem redundância):**
- Tabelas espaciais do DuckDB → camadas vetoriais
- Tabelas alfanuméricas do DuckDB → tabelas SQLite no mesmo arquivo
- Rasters `*_recortado.tif` em `data/raw/{municipio}/` → camadas raster

Os arquivos brutos `.gpkg` e `.csv` em `data/raw/` são **ignorados** — seu conteúdo já está no DuckDB.

---
**Fluxo:** execute a célula 1 para ver os municípios disponíveis, depois configure `MUNICIPIO` na célula 2 e execute.

In [2]:
# ── Célula 1 — Municípios disponíveis ────────────────────────────────────────
from pathlib import Path

BASE_DIR      = Path("data")
processed_dir = BASE_DIR / "processed"
raw_dir_base  = BASE_DIR / "raw"

if not processed_dir.exists():
    print(f"Pasta não encontrada: {processed_dir.resolve()}")
else:
    municipios = sorted(
        d.name for d in processed_dir.iterdir()
        if d.is_dir() and any(d.glob("*.duckdb"))
    )

    if not municipios:
        print("Nenhum município processado encontrado.")
    else:
        print(f"{'#':<4}  {'Município':<35}  {'DuckDB':<30}  Grupos raw")
        print("─" * 85)
        for i, nome in enumerate(municipios, 1):
            db   = next((processed_dir / nome).glob("*.duckdb"))
            raw  = raw_dir_base / nome
            grupos_raw = sorted(p.name for p in raw.iterdir() if p.is_dir()) if raw.exists() else []
            rasters    = list(raw.rglob("*_recortado.tif")) if raw.exists() else []
            print(
                f"{i:<4}  {nome:<35}  {db.name:<30}  "
                f"{grupos_raw}  ({len(rasters)} raster(s))"
            )

#     Município                            DuckDB                          Grupos raw
─────────────────────────────────────────────────────────────────────────────────────
1     al_arapiraca                         al_arapiraca.duckdb             ['censo', 'geometria', 'logradouros', 'luminosidade', 'pnadc']  (1 raster(s))
2     al_campo_alegre                      al_campo_alegre.duckdb          ['censo', 'geometria', 'logradouros', 'luminosidade', 'pnadc']  (1 raster(s))
3     al_carneiros                         al_carneiros.duckdb             ['geometria']  (0 raster(s))


---
## Configuração
Altere `MUNICIPIO` abaixo para o nome exibido na célula anterior e execute.

In [3]:
# ── Célula 2 — Exportar para GeoPackage ──────────────────────────────────────
import logging
import sqlite3
from pathlib import Path

import duckdb
import geopandas as gpd
import pandas as pd
from shapely import wkb as shapely_wkb

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# ╔══════════════════════════════════════════════════════╗
# ║  CONFIGURAR AQUI                                     ║
MUNICIPIO = "al_arapiraca"
BASE_DIR  = Path("data")
# ╚══════════════════════════════════════════════════════╝

raw_dir       = BASE_DIR / "raw"       / MUNICIPIO
processed_dir = BASE_DIR / "processed" / MUNICIPIO
output_gpkg   = processed_dir / f"{MUNICIPIO}_export.gpkg"

db_path = next(processed_dir.glob("*.duckdb"), None)
if db_path is None:
    raise FileNotFoundError(f"Nenhum .duckdb encontrado em {processed_dir}")

if output_gpkg.exists():
    output_gpkg.unlink()
    logger.info("Arquivo anterior removido: %s", output_gpkg.name)

# ─── 1. Inventário das fontes ────────────────────────────────────────────────
conn = duckdb.connect(str(db_path), read_only=True)
conn.execute("INSTALL spatial; LOAD spatial;")

todas_tabelas = conn.execute(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_schema = 'main' ORDER BY table_name"
).fetchdf()["table_name"].tolist()

# Tabelas que têm coluna 'geometry' (WKB) → camadas vetoriais
tab_espaciais = conn.execute(
    "SELECT DISTINCT table_name FROM information_schema.columns "
    "WHERE column_name = 'geometry' AND table_schema = 'main' "
    "ORDER BY table_name"
).fetchdf()["table_name"].tolist()

# Tabelas sem geometry → tabelas alfanuméricas
tab_alfa = [t for t in todas_tabelas if t not in tab_espaciais]

# Rasters processados (clipped) — não estão no DuckDB como geometria
rasters = sorted(raw_dir.rglob("*_recortado.tif")) if raw_dir.exists() else []

print(f"\nFontes encontradas — {MUNICIPIO}")
print(f"  DuckDB         : {db_path.name}")
print(f"  Tabelas espaciais ({len(tab_espaciais)}) : {tab_espaciais}")
print(f"  Tabelas alfanum.  ({len(tab_alfa)}) : {tab_alfa}")
print(f"  Rasters           ({len(rasters)}) : {[r.name for r in rasters]}")
print()

# ─── 2. Tabelas espaciais → camadas vetoriais do GeoPackage ──────────────────
primeiro_layer = True
n_espaciais = 0
for tabela in tab_espaciais:
    try:
        df = conn.execute(f'SELECT * FROM "{tabela}"').fetchdf()
        # WKB bytes → shapely geometry
        df["geometry"] = df["geometry"].apply(
            lambda b: shapely_wkb.loads(bytes(b)) if b is not None else None
        )
        gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4674")
        modo = "w" if primeiro_layer else "a"
        gdf.to_file(output_gpkg, layer=tabela, driver="GPKG", mode=modo)
        primeiro_layer = False
        n_espaciais += 1
        logger.info("✓ Espacial: %-35s  %d feições", tabela, len(gdf))
    except Exception as exc:
        logger.warning("✗ Espacial '%s': %s", tabela, exc)

# ─── 3. Tabelas alfanuméricas → tabelas SQLite no GeoPackage ─────────────────
# GeoPackage é SQLite; tabelas sem geometria ficam acessíveis via DB Manager do QGIS
n_alfa = 0
if tab_alfa:
    with sqlite3.connect(str(output_gpkg)) as lite:
        for tabela in tab_alfa:
            try:
                df = conn.execute(f'SELECT * FROM "{tabela}"').fetchdf()
                df.to_sql(tabela, lite, if_exists="replace", index=False)
                n_alfa += 1
                logger.info("✓ Alfanum.:  %-35s  %d linhas", tabela, len(df))
            except Exception as exc:
                logger.warning("✗ Alfanum. '%s': %s", tabela, exc)

conn.close()

# ─── 4. Rasters → camadas raster do GeoPackage ───────────────────────────────
n_rasters = 0
if rasters:
    try:
        from osgeo import gdal
        gdal.UseExceptions()
        gdal_ok = True
    except ImportError:
        logger.warning(
            "osgeo.gdal não disponível — %d raster(s) não exportado(s). "
            "Instale com: conda install -c conda-forge gdal",
            len(rasters),
        )
        gdal_ok = False

    if gdal_ok:
        for tif in rasters:
            # Nome da camada: subpasta(s) + stem, ex: luminosidade_viirs_2022_recortado
            partes = tif.relative_to(raw_dir).with_suffix("").parts
            layer  = "_".join(partes)
            try:
                src  = gdal.Open(str(tif))
                opts = gdal.TranslateOptions(
                    format="GPKG",
                    creationOptions=[
                        f"RASTER_TABLE={layer}",
                        "APPEND_SUBDATASET=YES",
                    ],
                )
                gdal.Translate(str(output_gpkg), src, options=opts)
                src = None
                n_rasters += 1
                logger.info("✓ Raster:    %-35s  %s", layer, tif.name)
            except Exception as exc:
                logger.warning("✗ Raster '%s': %s", tif.name, exc)

# ─── 5. Resumo ────────────────────────────────────────────────────────────────
tamanho_mb = output_gpkg.stat().st_size / 1_048_576
print()
print("═" * 60)
print(f"GeoPackage gerado : {output_gpkg}")
print(f"Tamanho           : {tamanho_mb:.1f} MB")
print(f"Camadas vetoriais : {n_espaciais}")
print(f"Tabelas alfanum.  : {n_alfa}")
print(f"Camadas raster    : {n_rasters}")
print("═" * 60)
print()
print("Para abrir no QGIS: Camada → Adicionar camada → Adicionar camada de GeoPackage")
print("Tabelas alfanuméricas: menu Banco de dados → Gerenciador BD → navegue até o arquivo")


Fontes encontradas — al_arapiraca
  DuckDB         : al_arapiraca.duckdb
  Tabelas espaciais (8) : ['areas_ponderacao', 'eixos_osm', 'enderecos_cnefe', 'faces_logradouro', 'grade_estatistica', 'limite_municipal', 'limite_municipal_teste', 'setores_censitarios']
  Tabelas alfanum.  (8) : ['censo_domicilio01', 'censo_domicilio02', 'censo_responsavel01', 'info_municipio', 'luminosidade_2022', 'luminosidade_2022_grade200', 'pnadc_estimativas', 'pnadc_metadados']
  Rasters           (1) : ['viirs_2022_recortado.tif']



18:55:25 [INFO] Created 12 records
18:55:25 [INFO] ✓ Espacial: areas_ponderacao                     12 feições
18:55:27 [INFO] Created 27,153 records
18:55:28 [INFO] ✓ Espacial: eixos_osm                            27153 feições
18:55:31 [INFO] Created 120,326 records
18:55:33 [INFO] ✓ Espacial: enderecos_cnefe                      120326 feições
18:55:33 [INFO] Created 17,460 records
18:55:35 [INFO] ✓ Espacial: faces_logradouro                     17460 feições
18:55:36 [INFO] Created 4,940 records
18:55:38 [INFO] ✓ Espacial: grade_estatistica                    4940 feições
18:55:38 [INFO] Created 1 records
18:55:41 [INFO] ✓ Espacial: limite_municipal                     1 feições
18:55:41 [INFO] Created 1 records
18:55:43 [INFO] ✓ Espacial: limite_municipal_teste               1 feições
18:55:44 [INFO] Created 396 records
18:55:46 [INFO] ✓ Espacial: setores_censitarios                  396 feições
18:55:47 [INFO] ✓ Alfanum.:  censo_domicilio01                    426 linhas
18:55:48 


════════════════════════════════════════════════════════════
GeoPackage gerado : data\processed\al_arapiraca\al_arapiraca_export.gpkg
Tamanho           : 28.5 MB
Camadas vetoriais : 8
Tabelas alfanum.  : 8
Camadas raster    : 0
════════════════════════════════════════════════════════════

Para abrir no QGIS: Camada → Adicionar camada → Adicionar camada de GeoPackage
Tabelas alfanuméricas: menu Banco de dados → Gerenciador BD → navegue até o arquivo
